# 15 - Opt-in executor-partition CPU inference prototype

This notebook is an experimental CPU-only Spark `mapPartitions` prototype. It is not part of the production `04_process_video` lease/commit contract. It demonstrates whole-video partition processing, one executor-local runtime cache per Python worker, explicit CPU thread budgeting, compact result/error records, and controlled Delta persistence.

In [ ]:
TABLE_PREFIX = "pc"
DATABASE = ""
INPUT_TABLE = ""  # Defaults to {TABLE_PREFIX}_executor_partition_input
OUTPUT_TABLE = ""  # Defaults to {TABLE_PREFIX}_executor_partition_records
ACTIVE_TASKS_PER_EXECUTOR = 4
EXECUTOR_CORES = 4
TARGET_PARTITIONS = 4
OUTPUT_TXN_APP_ID = "UNSET"
OUTPUT_TXN_VERSION = -1
FAIL_ON_ERRORS = True


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

from people_counter import RFDetrBotsortConfig, RTDetrOsnetConfig
from people_counter.fabric_executor_partition import (
    calculate_thread_budget,
    configure_cpu_runtime,
    executor_partition_schema,
    process_video_partition,
)


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
spark_session.conf.set("spark.sql.adaptive.enabled", "false")

if not TABLE_PREFIX.strip():
    raise ValueError("TABLE_PREFIX must be a non-empty string")
for name, value in {
    "EXECUTOR_CORES": EXECUTOR_CORES,
    "ACTIVE_TASKS_PER_EXECUTOR": ACTIVE_TASKS_PER_EXECUTOR,
    "TARGET_PARTITIONS": TARGET_PARTITIONS,
}.items():
    if type(value) is not int or value < 1:
        raise ValueError(f"{name} must be a positive integer")
if not OUTPUT_TXN_APP_ID.strip() or OUTPUT_TXN_APP_ID.strip().upper() == "UNSET":
    raise ValueError("OUTPUT_TXN_APP_ID must identify this input batch")
if type(OUTPUT_TXN_VERSION) is not int or OUTPUT_TXN_VERSION < 0:
    raise ValueError("OUTPUT_TXN_VERSION must be a non-negative integer")

thread_budget = calculate_thread_budget(EXECUTOR_CORES, ACTIVE_TASKS_PER_EXECUTOR)
spark_session.conf.set("spark.task.cpus", str(thread_budget.threads_per_worker))
print(
    {
        "executor_cores": thread_budget.driver_cores,
        "active_tasks_per_executor": thread_budget.active_workers,
        "threads_per_worker": thread_budget.threads_per_worker,
    }
)


In [ ]:
def table(suffix: str) -> str:
    prefix = f"{DATABASE}." if DATABASE else ""
    return f"{prefix}{TABLE_PREFIX}_{suffix}"


input_table = INPUT_TABLE or table("executor_partition_input")
output_table = OUTPUT_TABLE or table("executor_partition_records")

# Expected prepared-input columns:
# work_id, attempt_id, local_video_path, pipeline, device_variant, device,
# batch_size, sample_fps, detection_threshold, use_fp16, line,
# detector_model, camera_motion_compensation, models_dir.
# Keep one row per whole video; do not pre-split frames.
prepared = (
    spark_session.table(input_table)
    .where(F.col("local_video_path").isNotNull())
    .repartition(TARGET_PARTITIONS, F.col("work_id"))
)
prepared.select("work_id", "local_video_path", "pipeline").show(10, truncate=False)


In [ ]:
def build_config(row):
    video = Path(row["local_video_path"])
    if row.get("device_variant", "cpu") != "cpu" or row.get("device", "cpu") != "cpu":
        raise ValueError("Executor-partition prototype supports only CPU inference")
    if not row.get("models_dir"):
        raise ValueError("models_dir is required for offline executor inference")
    models_dir = Path(row["models_dir"])
    line = tuple(row["line"]) if row.get("line") else None
    common = {
        "video": video,
        "device_variant": row.get("device_variant", "cpu"),
        "device": row.get("device", "cpu"),
        "batch_size": int(row.get("batch_size", 1)),
        "sample_fps": row.get("sample_fps", 3.0),
        "detection_threshold": float(row.get("detection_threshold", 0.6)),
        "use_fp16": bool(row.get("use_fp16", False)),
        "line": line,
        "models_dir": models_dir,
    }
    if row.get("pipeline") == "rtdetr-osnet":
        return RTDetrOsnetConfig(
            **common,
            detector_model=row.get("detector_model", "r18"),
        )
    if row.get("pipeline") == "rfdetr-botsort":
        return RFDetrBotsortConfig(
            **common,
            camera_motion_compensation=row.get("camera_motion_compensation"),
        )
    raise ValueError(f"Unsupported pipeline: {row.get('pipeline')}")


def partition_records(rows):
    configure_cpu_runtime(EXECUTOR_CORES, ACTIVE_TASKS_PER_EXECUTOR)
    dictionaries = (row.asDict(recursive=True) for row in rows)
    yield from process_video_partition(dictionaries, config_builder=build_config)


records_rdd = prepared.rdd.mapPartitions(partition_records)
records = spark_session.createDataFrame(records_rdd, executor_partition_schema())
records.cache()
records.groupBy("record_type", "status").count().show(truncate=False)


In [ ]:
# Persist compact records with a stable Delta transaction id. A productionized
# version should MERGE these records into attempt-scoped output tables and use
# the existing control writer rather than appending directly from this notebook.
(
    records.write.format("delta")
    .mode("append")
    .option("txnAppId", OUTPUT_TXN_APP_ID)
    .option("txnVersion", int(OUTPUT_TXN_VERSION))
    .saveAsTable(output_table)
)

persisted = spark_session.table(output_table).where(
    F.col("emitted_at_utc").isNotNull()
)
persisted.groupBy("record_type", "status").count().show(truncate=False)


In [ ]:
summary = {row["status"]: row["count"] for row in records.groupBy("status").count().collect()}
error_count = int(summary.get("FAILED", 0))
success_count = int(summary.get("SUCCEEDED", 0))
print({"succeeded_records": success_count, "failed_records": error_count})
if FAIL_ON_ERRORS and error_count:
    records.where(F.col("status") == "FAILED").select(
        "work_id", "error_type", "error_message", "payload_json"
    ).show(20, truncate=False)
    raise RuntimeError(f"Executor-partition prototype produced {error_count} failed records")
if success_count == 0:
    raise RuntimeError("Executor-partition prototype produced no successful records")
